# Free Local Hybrid RAG + Reranking

This notebook demonstrates a complete retrieval pipeline using **free/local models**:

**Documents → Vector Search + BM25 → RRF → Local Reranker → Local LLM**

No OpenAI API, Cohere API, or paid model is used.

> **Learning goal:** Run each cell from top to bottom and inspect the results after every stage.

## Cell 1 — Install free libraries

This installs the open-source libraries needed for embeddings, vector search, BM25,
reranking, and local text generation.

In [1]:
# Install only free/open-source packages.
# No API key is required.

%pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    rank_bm25 \
    sentence-transformers \
    transformers \
    accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/

## Cell 2 — Import libraries

These imports provide:
- `Document` for storing text chunks
- Hugging Face embeddings for semantic search
- Chroma for the local vector database
- BM25 for keyword retrieval
- Sentence Transformers for the local reranker
- Transformers for the final local LLM

In [2]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever

from sentence_transformers import CrossEncoder

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import torch

print("✅ Imports successful")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/tmp/ipykernel_1484/664593134.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


✅ Imports successful
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Cell 3 — Create sample documents

These are small demo chunks so that the retrieval process is easy to understand.

Later, replace this list with the real chunks from your RAG project.

In [3]:
chunks = [
    # Tesla
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",

    # Microsoft
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",

    # NVIDIA
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",

    # Google
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",

    # Noise / less relevant examples
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA graphics cards are also popular among PC gamers.",
    "Google Maps provides navigation and location services.",
]

print(f"Created {len(chunks)} sample chunks.")

Created 44 sample chunks.


## Cell 4 — Convert chunks to Documents

LangChain's `Document` object stores the text together with metadata.

The metadata gives every chunk a simple ID that we can use when comparing retrieval results.

In [4]:
documents = [
    Document(
        page_content=chunk,
        metadata={"source": f"chunk_{i}"}
    )
    for i, chunk in enumerate(chunks)
]

print(f"Created {len(documents)} Document objects.")
print("\nExample:")
print(documents[0])

Created 44 Document objects.

Example:
page_content='Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.' metadata={'source': 'chunk_0'}


## Cell 5 — Load a free embedding model

`all-MiniLM-L6-v2` is a small, free Sentence Transformers model.

It converts text into vectors so that vector search can compare the **meaning** of queries and documents.

In [5]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

print(f"✅ Loaded embedding model: {EMBEDDING_MODEL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


## Cell 6 — Create the local vector database

Chroma stores the document embeddings locally.

This is the **dense/semantic retrieval** part of the pipeline.

In [6]:
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 15}
)

print("✅ Local Chroma vector store created.")

✅ Local Chroma vector store created.


## Cell 7 — Test vector search

Vector search looks for documents that are semantically similar to the query.

Try changing the query and observe the results.

In [7]:
query = "Tesla financial performance and production updates"

vector_results = vector_retriever.invoke(query)

print("QUERY:", query)
print("\nVECTOR SEARCH RESULTS:")

for i, doc in enumerate(vector_results[:10], start=1):
    print(f"{i:2d}. {doc.page_content}")

QUERY: Tesla financial performance and production updates

VECTOR SEARCH RESULTS:
 1. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 2. Tesla's automotive gross margin improved to 19.3% this quarter.
 3. Tesla's energy storage business grew 40% year-over-year.
 4. Tesla reported strong free cash flow generation of $7.5 billion.
 5. Tesla's Full Self-Driving revenue increased significantly.
 6. Tesla announced plans to expand Gigafactory production capacity.
 7. Tesla stock price reached new highs following earnings announcement.
 8. Tesla continues to lead in electric vehicle market share globally.
 9. Tesla Model Y became the best-selling vehicle worldwide.
10. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.


## Cell 8 — Create BM25 retriever

BM25 is a **keyword-based/sparse retrieval** method.

It is useful when exact terms, names, product names, or numbers matter.

In [8]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 15

print("✅ BM25 retriever created.")

✅ BM25 retriever created.


## Cell 9 — Test BM25

This runs the same query through BM25.

Compare its results with the vector-search results from the previous cell.

In [9]:
bm25_results = bm25_retriever.invoke(query)

print("QUERY:", query)
print("\nBM25 RESULTS:")

for i, doc in enumerate(bm25_results[:10], start=1):
    print(f"{i:2d}. {doc.page_content}")

QUERY: Tesla financial performance and production updates

BM25 RESULTS:
 1. Tesla announced plans to expand Gigafactory production capacity.
 2. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 3. Google Maps provides navigation and location services.
 4. The Tesla coil was invented by Nikola Tesla in 1891.
 5. Tesla Model Y became the best-selling vehicle worldwide.
 6. Tesla stock price reached new highs following earnings announcement.
 7. Tesla reported strong free cash flow generation of $7.5 billion.
 8. Tesla continues to lead in electric vehicle market share globally.
 9. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
10. Google faces regulatory scrutiny over AI dominance.


## Cell 10 — Reciprocal Rank Fusion (RRF)

RRF combines ranked lists from different retrieval methods.

For each document:

`RRF score = 1 / (k + rank)`

A document that appears near the top of multiple retrievers receives a stronger combined score.

`k=60` is a common smoothing value.

In [10]:
def reciprocal_rank_fusion(result_lists, k=60):
    """Combine multiple ranked document lists using RRF."""

    scores = {}
    documents_by_id = {}

    for results in result_lists:
        for rank, document in enumerate(results, start=1):

            # Metadata source is a stable ID for our demo documents.
            doc_id = document.metadata.get(
                "source",
                document.page_content
            )

            # RRF formula.
            score = 1.0 / (k + rank)

            scores[doc_id] = scores.get(doc_id, 0.0) + score
            documents_by_id[doc_id] = document

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        (documents_by_id[doc_id], scores[doc_id])
        for doc_id in ranked_ids
    ]

print("✅ RRF function ready.")

✅ RRF function ready.


## Cell 11 — Run RRF

This is the first true **hybrid retrieval** stage:

**Vector Search + BM25 → RRF → combined ranking**

In [11]:
rrf_results = reciprocal_rank_fusion(
    [
        vector_results,
        bm25_results
    ],
    k=60
)

print("RRF RESULTS:")

for rank, (doc, score) in enumerate(rrf_results[:10], start=1):
    print(f"{rank:2d}. RRF score={score:.5f} | {doc.page_content}")

RRF RESULTS:
 1. RRF score=0.03154 | Tesla announced plans to expand Gigafactory production capacity.
 2. RRF score=0.03089 | Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 3. RRF score=0.03055 | Tesla reported strong free cash flow generation of $7.5 billion.
 4. RRF score=0.03041 | Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 5. RRF score=0.03008 | Tesla stock price reached new highs following earnings announcement.
 6. RRF score=0.02988 | Tesla Model Y became the best-selling vehicle worldwide.
 7. RRF score=0.02971 | The Tesla coil was invented by Nikola Tesla in 1891.
 8. RRF score=0.02941 | Tesla continues to lead in electric vehicle market share globally.
 9. RRF score=0.01613 | Tesla's automotive gross margin improved to 19.3% this quarter.
10. RRF score=0.01587 | Tesla's energy storage business grew 40% year-over-year.


## Cell 12 — Free local reranker

The original notebook used **Cohere Rerank**, which requires a Cohere API/service.

We replace it with:

`BAAI/bge-reranker-base`

This is a free, locally loaded cross-encoder reranker from Hugging Face.

It receives the query and candidate documents and gives each candidate a relevance score.

In [12]:
RERANKER_MODEL = "BAAI/bge-reranker-base"

reranker = CrossEncoder(
    RERANKER_MODEL,
    max_length=512,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print(f"✅ Loaded local reranker: {RERANKER_MODEL}")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

✅ Loaded local reranker: BAAI/bge-reranker-base


## Cell 13 — Rerank the RRF results

RRF is good at combining different retrieval systems.

The reranker then performs a more detailed query-document relevance comparison.

We first take the top 15 RRF candidates and rerank them.

In [13]:
# Take the top candidates from RRF before expensive reranking.
candidate_docs = [doc for doc, score in rrf_results[:15]]

pairs = [
    (query, doc.page_content)
    for doc in candidate_docs
]

rerank_scores = reranker.predict(
    pairs,
    show_progress_bar=True
)

reranked = sorted(
    zip(candidate_docs, rerank_scores),
    key=lambda item: float(item[1]),
    reverse=True
)

reranked_docs = [doc for doc, score in reranked]

print("RERANKED RESULTS:")

for rank, (doc, score) in enumerate(reranked[:10], start=1):
    print(f"{rank:2d}. score={float(score):.4f} | {doc.page_content}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RERANKED RESULTS:
 1. score=0.2026 | Tesla's automotive gross margin improved to 19.3% this quarter.
 2. score=0.1957 | Tesla reported strong free cash flow generation of $7.5 billion.
 3. score=0.1626 | Tesla stock price reached new highs following earnings announcement.
 4. score=0.0571 | Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 5. score=0.0517 | Tesla announced plans to expand Gigafactory production capacity.
 6. score=0.0449 | Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 7. score=0.0378 | Tesla continues to lead in electric vehicle market share globally.
 8. score=0.0256 | Tesla's Full Self-Driving revenue increased significantly.
 9. score=0.0196 | Tesla's energy storage business grew 40% year-over-year.
10. score=0.0021 | Tesla Model Y became the best-selling vehicle worldwide.


## Cell 14 — Compare before and after reranking

This cell makes the purpose of reranking visible.

The RRF list is the ranking before the cross-encoder.
The reranked list is the ranking after the cross-encoder.

In [14]:
print("=" * 80)
print("BEFORE RERANKING — RRF")
print("=" * 80)

for i, (doc, score) in enumerate(rrf_results[:5], start=1):
    print(f"{i}. {doc.page_content}")

print("\n" + "=" * 80)
print("AFTER RERANKING — BGE RERANKER")
print("=" * 80)

for i, (doc, score) in enumerate(reranked[:5], start=1):
    print(f"{i}. score={float(score):.4f} | {doc.page_content}")

BEFORE RERANKING — RRF
1. Tesla announced plans to expand Gigafactory production capacity.
2. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
3. Tesla reported strong free cash flow generation of $7.5 billion.
4. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
5. Tesla stock price reached new highs following earnings announcement.

AFTER RERANKING — BGE RERANKER
1. score=0.2026 | Tesla's automotive gross margin improved to 19.3% this quarter.
2. score=0.1957 | Tesla reported strong free cash flow generation of $7.5 billion.
3. score=0.1626 | Tesla stock price reached new highs following earnings announcement.
4. score=0.0571 | Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
5. score=0.0517 | Tesla announced plans to expand Gigafactory production capacity.


## Cell 15 — Load a free local language model

The original notebook used OpenAI `gpt-4o`, which requires a paid API.

We replace it with:

`Qwen/Qwen2.5-0.5B-Instruct`

This is a small instruction-tuned model that can run locally. On a T4 it is suitable for learning and testing.

If generation is slow or you run out of memory, this cell can be run on CPU instead, but CPU generation will be slower.

In [15]:
GENERATION_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    "text-generation",
    model=GENERATION_MODEL,
    tokenizer=GENERATION_MODEL,
    device=device,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False
)

print(f"✅ Loaded local generation model: {GENERATION_MODEL}")
print("Generation device:", "GPU" if device == 0 else "CPU")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Loaded local generation model: Qwen/Qwen2.5-0.5B-Instruct
Generation device: GPU


## Cell 16 — Build the RAG prompt

The top reranked documents become the context for the language model.

The model is instructed to answer using only that context.

In [16]:
top_reranked = reranked_docs[:5]

context = "\n".join(
    f"- {doc.page_content}"
    for doc in top_reranked
)

prompt = f"""You are a helpful RAG assistant.

Answer the question using only the information in the context.

If the context does not contain enough information, say:
"I don't have enough information in the provided context."

Question:
{query}

Context:
{context}

Answer:
"""

print(prompt)

You are a helpful RAG assistant.

Answer the question using only the information in the context.

If the context does not contain enough information, say:
"I don't have enough information in the provided context."

Question:
Tesla financial performance and production updates

Context:
- Tesla's automotive gross margin improved to 19.3% this quarter.
- Tesla reported strong free cash flow generation of $7.5 billion.
- Tesla stock price reached new highs following earnings announcement.
- Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
- Tesla announced plans to expand Gigafactory production capacity.

Answer:



## Cell 17 — Generate the final answer

This is the final RAG stage:

**Query → Retrieval → RRF → Reranking → Context → Local LLM → Answer**

In [17]:
result = generator(prompt)

answer = result[0]["generated_text"].strip()

print("=" * 80)
print("FINAL RAG ANSWER")
print("=" * 80)
print(answer)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


FINAL RAG ANSWER
Tesla's financial performance and production updates include:

- Improved automotive gross margin at 19.3%
- Strong free cash flow generation of $7.5 billion
- Record quarterly revenue of $25.2 billion in Q3 2024
- Plans to expand Gigafactory production capacity

I don't have enough information in the provided context. The context focuses on Tesla's financial performance and production updates, but it doesn't provide specific details about its current market position or future plans. Therefore, I cannot answer the question based solely on the given context. If you need more detailed information about Tesla's financial performance and production, I would recommend checking their latest financial reports or news articles for that specific update. However, the context already provides valuable insights into Tesla's current status and upcoming developments. Let me know if you'd like any other information from the context!


## Cell 18 — Change the query and experiment

Try different questions.

The important thing is to observe how the results change at each stage:
1. Vector search
2. BM25
3. RRF
4. Reranking
5. Final generation

In [18]:
query = "What are NVIDIA's AI hardware and data center developments?"

# Vector retrieval
vector_results = vector_retriever.invoke(query)

# BM25 retrieval
bm25_results = bm25_retriever.invoke(query)

# RRF fusion
rrf_results = reciprocal_rank_fusion(
    [vector_results, bm25_results],
    k=60
)

# Reranking
candidate_docs = [doc for doc, score in rrf_results[:15]]

pairs = [
    (query, doc.page_content)
    for doc in candidate_docs
]

scores = reranker.predict(
    pairs,
    show_progress_bar=False
)

reranked = sorted(
    zip(candidate_docs, scores),
    key=lambda item: float(item[1]),
    reverse=True
)

reranked_docs = [doc for doc, score in reranked]

# Build context
context = "\n".join(
    f"- {doc.page_content}"
    for doc in reranked_docs[:5]
)

prompt = f"""Answer the question using only the context.

Question:
{query}

Context:
{context}

Answer:
"""

result = generator(prompt)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(result[0]["generated_text"].strip())

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What are NVIDIA's AI hardware and data center developments?

ANSWER:
NVIDIA has made significant advancements in its AI hardware and data center developments. The company's CUDA platform, which dominates AI development, continues to dominate the market. In addition to this, NVIDIA's automotive AI platform partnerships have expanded, further strengthening its position in the industry. Furthermore, NVIDIA's partnership with major cloud providers has strengthened its presence in the AI data center market. Finally, NVIDIA's H100 GPUs have seen unprecedented demand for AI training due to their superior performance compared to other GPUs. Lastly, NVIDIA announced new AI inference chips for edge computing, expanding its offerings in the field of AI applications. These developments demonstrate NVIDIA's commitment to advancing AI technology and improving the efficiency of AI systems.


# Final architecture

The notebook now uses only free/local models:

```text
Documents
   ↓
all-MiniLM-L6-v2
   ↓
Vector Search ──────┐
                    │
BM25 ───────────────┤
                    ↓
                   RRF
                    ↓
          BGE Reranker (local)
                    ↓
             Top relevant chunks
                    ↓
       Qwen2.5-0.5B-Instruct (local)
                    ↓
               Final answer
```

## Paid services removed

- OpenAI `text-embedding-3-small` → `all-MiniLM-L6-v2`
- Cohere `rerank-english-v3.0` → `BAAI/bge-reranker-base`
- OpenAI `gpt-4o` → `Qwen/Qwen2.5-0.5B-Instruct`

No API key is required.